<a href="https://colab.research.google.com/github/danielbrodev/ai_foundation/blob/main/menu_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("Menu Detector!")

Menu Detector!


#  Menu detector loyihasi bu taomlar rasmlari orqali train qilib malum taomlarni taniydigan model

In [ ]:
#---------------------
# Import Libraries
#---------------------

from google.colab import drive
import torchvision.transforms as transforms

from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset
import os
import numpy as np

In [ ]:
drive.mount('/content/drive') # drive ga ulanamiz

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Define dataset path
DATASET_PATH = '/content/drive/MyDrive/food-dataset'


CUSTOM_CLASS_MAPPING = {
    'hamburger': 'hamburger',
    'hot_dog': 'hot_dog',
    'chocolate_cake': 'dessert', # label grouping  |  class consolidation
    'kebab': 'kebab',
    'cheesecake': 'dessert',
    'pilaf': 'pilaf',
    'ice_cream': 'dessert'     # label grouping  |  class consolidation
}





CLASSES = ['hamburger', 'dessert', 'hot_dog', 'kebab', 'pilaf' ]
CLASS_TO_INDEX = {c: i for i, c in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)

print(f'Classes: {CLASSES}')
print(f'Class to index: {CLASS_TO_INDEX}')
print(f'Number of classes: {NUM_CLASSES}')

# bu bizning ishlarimiz ketmaketligini belgilaydi
transform = transforms.Compose([
    transforms.Resize((224, 224)) , # bu rasmlarimiz sizeni o'zgartiradi
    transforms.ToTensor(), # tensorga o'giramiz(raqamlarga)
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
# Tensorga o'girgach u flot korinishiga otadi 0.0 ~ 1.0
# H, W, C => C, H, W  | => channel : RGB
# Normalize => hamma channellarni qayta sozlab beradi
# Normalize : pixel = (pixel-mean)/std



print("transform:", transform)

Classes: ['hamburger', 'dessert', 'hot_dog', 'kebab', 'pilaf']
Class to index: {'hamburger': 0, 'dessert': 1, 'hot_dog': 2, 'kebab': 3, 'pilaf': 4}
Number of classes: 5
transform: Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


In [ ]:
# -------------------------
# Custom Dataset Class
# -------------------------

class FoodDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform
# umumiy filelarni o'qish vaqtida data uzunligini olib beradi
    def __len__(self):
        print('images_length', len(self.images))
        return len(self.images)
#
    def __getitem__(self, idx):
        img_path = self.images[idx]
        print('image_path', img_path)

        label = self.labels[idx]
        print('label', label)
# agar rasmda RGB bolmasa convert qilamiz
        try:
            image = Image.open(img_path).convert('RGB')
        except (UnidentifiedImageError, OSError):
            print(f"Skipping broken image: {img_path}") # rasm filelarda kamchilik bolsa skip qilish
            return self.__getitem__((idx + 1) % len(self.images))

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
# -------------------------
# Gather and Split Data
# -------------------------

all_images = [] # created empty list

for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():
    class_path = os.path.join(DATASET_PATH, original_class)  # /content/drive/MyDrive/food-dataset/hamburber
    print('class_path:', class_path)

    if not os.path.exists(class_path):
        print(f"Warning: {class_path} not found")
        continue

    for img in os.listdir(class_path):
        if img.endswith(('.jpg', '.jpeg', '.png')):
            full_path = os.path.join(class_path, img) # /content/drive/MyDrive/food-dataset/hamburber/233.jpg
            all_images.append((full_path, CLASS_TO_INDEX[mapped_class])) # Changed CLASS_TO_IDX to CLASS_TO_INDEX

np.random.shuffle(all_images) #random aralashtiradi

split = int(0.8 * len(all_images))
train_data = all_images[:split] # 1000m | 800 train_data | 200 val_data
val_data = all_images[split:]

train_images, train_labels = zip(*train_data)
val_images, val_labels = zip(*val_data)

print('all_images:', all_images)

dataset = FoodDataset(train_images, train_labels)
print(len(dataset))
img, lbl = dataset[0]

class_path: /content/drive/MyDrive/food-dataset/hamburger
class_path: /content/drive/MyDrive/food-dataset/hot_dog
class_path: /content/drive/MyDrive/food-dataset/chocolate_cake
class_path: /content/drive/MyDrive/food-dataset/kebab
class_path: /content/drive/MyDrive/food-dataset/cheesecake
class_path: /content/drive/MyDrive/food-dataset/pilaf
class_path: /content/drive/MyDrive/food-dataset/ice_cream
all_images: [('/content/drive/MyDrive/food-dataset/chocolate_cake/Image_34.jpg', 1), ('/content/drive/MyDrive/food-dataset/pilaf/Image_2.jpeg', 4), ('/content/drive/MyDrive/food-dataset/pilaf/Image_60.jpg', 4), ('/content/drive/MyDrive/food-dataset/ice_cream/Image_24.jpg', 1), ('/content/drive/MyDrive/food-dataset/hot_dog/Image_22.jpg', 2), ('/content/drive/MyDrive/food-dataset/kebab/Image_30.jpg', 3), ('/content/drive/MyDrive/food-dataset/chocolate_cake/Image_48.png', 1), ('/content/drive/MyDrive/food-dataset/chocolate_cake/Image_36.jpg', 1), ('/content/drive/MyDrive/food-dataset/hamburger/